# FER2013 Flat Array Model Training

## Emotion Face Classifier

This notebook performs data preprocessing and analysis with FER 2013 data only.

## NOTE: Metrics here are training metrics only, test metrics are more meaningful

In [2]:
import os
import numpy as np
import pandas as pd
import importlib

In [3]:
def check_directory_name(target_name) -> bool:
    """
    Check if the current directory name matches the target_name.
    If not, move up a directory and repeat the check.
    
    Args:
        target_name (str): The directory name to match.
        
    Returns:
        bool: True if the current directory name matches the target_name, False otherwise.
    """
    # Get the current directory path
    current_dir = os.getcwd()
    
    # Extract the directory name from the path
    current_dir_name = os.path.basename(current_dir)
    
    # Check if the current directory name matches the target_name
    if current_dir_name == target_name:
        print(f'Directory set to {current_dir}, matches target dir sting {target_name}.')
        return True
    else:
        # Move up a directory
        os.chdir('..')
        # Check if we have reached the root directory
        if os.getcwd() == current_dir:
            return False
        # Recursively call the function to check the parent directory
        return check_directory_name(target_name)

In [4]:
main_dir = 'EmotionFaceClassifier'
check_directory_name(main_dir)

Directory set to /Users/dsl/Documents/GitHub/EmotionFaceClassifier, matches target dir sting EmotionFaceClassifier.


True

In [5]:
from src.main import (
    convert_pixels_to_array,
    str_to_array,
    load_config
)

## Data Import and Prep

In [7]:
df_path = 'data/fer2013/fer2013.csv'
df = pd.read_csv(df_path)

In [8]:
# Load common dicts from json config file
common_dicts = load_config('./configs/basics.json')
print(common_dicts.keys())

dict_keys(['usage_dict', 'emo_dict', 'emo_color_dict', 'output_col_order'])


In [9]:
# Load in key dicts from json for data mapping
emo_dict = common_dicts['emo_dict']
emo_color_dict = common_dicts['emo_color_dict']

In [10]:
### Optional, view dicts from json
# print(emo_dict)
# print('\n')
# print(emo_color_dict)

In [11]:
# Load model params from JSON
flat_models = load_config('./configs/flat_models.json')
print(flat_models.keys())

dict_keys(['LogisticRegression', 'DecisionTree', 'RandomForest', 'GradientBoosting', 'XGBoost', 'LGBM'])


In [12]:
### Optional, view model details
# print(flat_models)

In [13]:
# Modify df for clarity
df = df.rename(columns={'emotion': 'emotion_id'})
df['emotion'] = df['emotion_id'].astype(str).map(emo_dict)
df['color'] = df['emotion'].map(emo_color_dict)

In [14]:
# Pixel data is read in as str, must be converted to np.array 48x48
df['image'] = df['pixels'].apply(convert_pixels_to_array)

In [15]:
# Pixel data is read in as str, converted to a flat array (2304,)
df['flat_array'] = df['pixels'].apply(str_to_array)

## Flat Array Models

Trains each model specified in './configs/flat_models.json'

See flat_models variable for more details

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score
)

In [18]:
# Function to create model instances from the configuration
def create_models(details):
    model = {}
    for model_name, model_info in details.items():
        module = importlib.import_module(model_info['module'])
        model_class = getattr(module, model_info['class'])
        models[model_name] = model_class(**model_info['params'])
    return models

In [ ]:
# Function to create model instances from the configuration
def model_specs(detail_dict):
    for mdl_name, mdl_specs in detail_dict.items():
    # for model_name, model_info in details.items():
        module = importlib.import_module(mdl_specs['module'])
        model_class = getattr(module, mdl_specs['class'])
        mdl = model_class(**model_info['params'])
        # models[model_name] = model_class(**model_info['params'])
    return mdl

In [19]:
def classification_metrics(y_true, y_pred):
    metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted'),
            'recall': recall_score(y_test, y_pred, average='weighted'),
            'f1_score': f1_score(y_test, y_pred, average='weighted')
        }
    return metrics

In [20]:
def save_model(model, filename, library):
    if 'sklearn' in library:
        import joblib
        joblib.dump(model, filename)
    elif library == 'xgboost':
        model.save_model(filename)
    elif library == 'lightgbm':
        model.save_model(filename)
    elif library == 'keras':
        from tensorflow.keras.models import save_model
        save_model(model, filename)
    else:
        raise ValueError("Unsupported library")

# # Example usage
# save_model(sklearn_model, 'sklearn_model.pkl', library='sklearn')
# save_model(xgboost_model, 'xgboost_model.json', library='xgboost')
# save_model(lightgbm_model, 'lightgbm_model.txt', library='lightgbm')
# save_model(keras_model, 'keras_model.h5', library='keras')

In [21]:
flat_models

{'LogisticRegression': {'module': 'sklearn.linear_model',
  'class': 'LogisticRegression',
  'params': {'multi_class': 'multinomial', 'max_iter': 200}},
 'DecisionTree': {'module': 'sklearn.tree',
  'class': 'DecisionTreeClassifier',
  'params': {}},
 'RandomForest': {'module': 'sklearn.ensemble',
  'class': 'RandomForestClassifier',
  'params': {}},
 'GradientBoosting': {'module': 'sklearn.ensemble',
  'class': 'GradientBoostingClassifier',
  'params': {}},
 'XGBoost': {'module': 'xgboost',
  'class': 'XGBClassifier',
  'params': {'eval_metric': 'mlogloss'}},
 'LGBM': {'module': 'lightgbm', 'class': 'LGBMClassifier', 'params': {}}}

In [22]:
# def create_models(config):
models = create_models(flat_models)

In [23]:
results_df = pd.DataFrame()

In [24]:
X = np.stack(df['flat_array'].values)
y = df['emotion']

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    random_state=42,
    stratify=y
)

In [26]:
flat_model_dir = os.path.join('models', 'flat')
os.makedirs(flat_model_dir, exist_ok=True)

In [27]:
flat_models

{'LogisticRegression': {'module': 'sklearn.linear_model',
  'class': 'LogisticRegression',
  'params': {'multi_class': 'multinomial', 'max_iter': 200}},
 'DecisionTree': {'module': 'sklearn.tree',
  'class': 'DecisionTreeClassifier',
  'params': {}},
 'RandomForest': {'module': 'sklearn.ensemble',
  'class': 'RandomForestClassifier',
  'params': {}},
 'GradientBoosting': {'module': 'sklearn.ensemble',
  'class': 'GradientBoostingClassifier',
  'params': {}},
 'XGBoost': {'module': 'xgboost',
  'class': 'XGBClassifier',
  'params': {'eval_metric': 'mlogloss'}},
 'LGBM': {'module': 'lightgbm', 'class': 'LGBMClassifier', 'params': {}}}

In [28]:
# for model_name, model in models.items():
#     print(model_name)
#     print(model)
#     print(type(model))

In [29]:
for model_label, model_details in models.items():
    print(f"{model_label}: {model_details}")
    
    model_output_dir = os.path.join(flat_model_dir, model_label)
    os.makedirs(model_output_dir, exist_ok=True)
    
    mdl_output_path = os.path.join(model_output_dir, 'mdl.pkl')
    # mdl = model(random_state=42)
    model_details.fit(X_train, y_train)

    save_model(model_details, filename=model_output_dir, library=model_details['module'])

    model_preds = model_details.predict(X_test)

    mdl_results = get_classification_metrics(y_train, model_preds)

    cm_disp = ConfusionMatrixDisplay.from_predictions(
        y_true=y_train,
        y_pred=model_preds, 
        cmap='Blues'
    )    
    
    plt.tight_layout()
    plt.savefig(os.path.join(model_output_dir, 'cm_train.png'), pad_inches=5)


LogisticRegression: LogisticRegression(max_iter=200, multi_class='multinomial')


/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


TypeError: 'LogisticRegression' object is not subscriptable

In [ ]:
# dt = DecisionTreeClassifier(random_state=42)
# dt.fit(X_train, y_train)

In [ ]:
# dt_predictions = dt.predict(X_test)

In [ ]:
# dt_results = get_classification_metrics(y_test, dt_predictions)

In [ ]:
# dt_results

In [ ]:
# cm_disp = ConfusionMatrixDisplay.from_predictions(
#     y_true=y_test,
#     y_pred=dt_predictions, 
#     cmap='Blues'
# )

In [ ]:
# rf = RandomForestClassifier(n_estimators=25, random_state=42)
# rf.fit(X_train, y_train)

In [ ]:
# rf_predictions = rf.predict(X_test)

In [ ]:
# cm_disp = ConfusionMatrixDisplay.from_predictions(
#     y_true=y_test, 
#     y_pred=rf_predictions, 
#     cmap='Blues'
# )

In [ ]:
# rf_results = get_classification_metrics(y_test, rf_predictions)

### TODOs
Save metrics and cm
Use pre-selected train-test split

In [ ]:
# dt_results

In [ ]:
# rf_results